# 04. Monotonic Stack: Advanced Problems

Same stack, harder questions: **areas, contributions, and greedy digit removal.**

**Learning order:** HISTOGRAM → MAXIMAL RECTANGLE → TRAPPING WATER → SUM OF SUBARRAY MINIMUMS → REMOVE K DIGITS → INTERVIEW THINKING

## 1. Largest Rectangle in a Histogram

For each bar, the best rectangle that uses its **full height** extends left and right until a **shorter** bar. So we need, for every bar, the nearest smaller bar on both sides.

Keep an **increasing** stack of `(start_index, height)`. When a shorter bar arrives, pop taller bars: each popped bar's rectangle ends here, width = `i - start`. The new bar can **start** where the last popped bar started.

Real life: the largest billboard that fits under a skyline of buildings.

In [1]:
def largest_rectangle(heights):
    stack = []                                   # (start index, height)
    best = 0
    for i, h in enumerate(heights + [0]):        # sentinel 0 flushes the stack
        start = i
        while stack and stack[-1][1] > h:
            idx, height = stack.pop()
            best = max(best, height * (i - idx))
            start = idx
        stack.append((start, h))
    return best

print(largest_rectangle([2, 1, 5, 6, 2, 3]), largest_rectangle([2, 4]))   # 10 4

10 4


## 2. Maximal Rectangle in a binary matrix

Treat each row as the **base** of a histogram: `height[j]` = number of consecutive `1`s going up from this row. Then run the histogram solution for every row. O(rows × cols).

In [2]:
def maximal_rectangle(matrix):
    if not matrix:
        return 0
    heights = [0] * len(matrix[0])
    best = 0
    for row in matrix:
        for j, v in enumerate(row):
            heights[j] = heights[j] + 1 if v == "1" else 0
        best = max(best, largest_rectangle(heights))
    return best

grid = [["1", "0", "1", "0", "0"],
        ["1", "0", "1", "1", "1"],
        ["1", "1", "1", "1", "1"],
        ["1", "0", "0", "1", "0"]]
print(maximal_rectangle(grid))   # 6

6


## 3. Trapping Rain Water (stack version)

Keep a **decreasing** stack of bar indexes. When a taller bar `h` arrives, the popped bar is a "valley" bottom. The left wall is the new stack top, the right wall is `h`. Water layer = `width × (min(left, right) - bottom)`.

(The two-pointer solution from the Two Pointers topic is shorter. Know both.)

In [3]:
def trap(height):
    stack, water = [], 0
    for i, h in enumerate(height):
        while stack and h > height[stack[-1]]:
            bottom = stack.pop()
            if not stack:
                break
            left = stack[-1]
            width = i - left - 1
            bounded = min(height[left], h) - height[bottom]
            water += width * bounded
        stack.append(i)
    return water

print(trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))   # 6

6


## 4. Sum of Subarray Minimums (contribution technique)

Instead of enumerating all subarrays, ask: **for each element, in how many subarrays is it the minimum?**

If `left` = number of choices for the left end (until a strictly smaller element) and `right` = choices for the right end (until a smaller-or-equal element), element contributes `value × left × right`.

Using `>=` on one side and `>` on the other avoids double counting equal values. This "count contribution with nearest smaller on both sides" idea appears in many hard problems.

In [4]:
def sum_subarray_mins(arr):
    MOD = 10**9 + 7
    n = len(arr)
    stack, total = [], 0
    for i in range(n + 1):
        cur = arr[i] if i < n else float("-inf")
        while stack and arr[stack[-1]] >= cur:
            mid = stack.pop()
            left = mid - (stack[-1] if stack else -1)
            right = i - mid
            total += arr[mid] * left * right
        stack.append(i)
    return total % MOD

print(sum_subarray_mins([3, 1, 2, 4]), sum_subarray_mins([11, 81, 94, 43, 3]))   # 17 444

17 444


## 5. Remove K Digits (greedy + stack)

To make the number **smallest**, a digit should be removed when the next digit is **smaller** (it hurts the leading position). Keep an increasing stack, pop while `top > digit` and removals remain. If removals are left at the end, drop from the right. Strip leading zeros.

In [5]:
def remove_k_digits(num, k):
    stack = []
    for d in num:
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    if k:
        stack = stack[:-k]
    return "".join(stack).lstrip("0") or "0"

print(remove_k_digits("1432219", 3), remove_k_digits("10200", 1), remove_k_digits("10", 2))   # 1219 200 0

1219 200 0


## 6. Interview thinking

**Signals:** "largest rectangle/area", "sum over all subarrays of min/max", "smallest number after removing digits", "next greater with distances".

**Say:** "I need the nearest smaller on both sides for every element, so a monotonic stack gives that in O(n)."

**Common mistakes:** forgetting the sentinel to flush the stack, wrong width (`i - idx` vs `i - stack[-1] - 1`), double counting with equal values, leading zeros in Remove K Digits, not applying modulo.

## Quick Revision

- Histogram: increasing stack of `(start, height)`, width = `i - start`, sentinel 0 at the end.
- Maximal rectangle: heights per row + histogram.
- Trapping water: decreasing stack, `width × (min(walls) - bottom)`.
- Subarray minimums: contribution = `value × left × right`, `>=` on one side only.
- Remove k digits: pop larger tops while removals remain.